In [ ]:
%pip install scipy pandas scikit-learn tensorflow keras keras-tuner

import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV, PredefinedSplit
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn import metrics
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC

from keras import models, Input
from keras import optimizers as opt
from keras import backend as K
from keras.layers import Dense
from keras_tuner.tuners import RandomSearch
from tensorflow.keras.utils import to_categorical
from keras.optimizers import Adam
import importlib
import os
import numpy as np
import pandas as pd
import scipy
import variablesEEGMAT as v # this is where they are getting the test types and data types from


Note: you may need to restart the kernel to use updated packages.


Functions


In [ ]:
def load_dataset(data_type="raw", test_type="task"):
    '''
    Loads data from the EEG Physionet Dataset.
    
    Args:
        data_type (string): The data type to load. Defaults to "ica_filtered".
        test_type (string): The test type to load. Defaults to "Arithmetic".
    
    Returns:
        ndarray: The specified dataset.
    # NOTE: subtract eeg background from eeg task data
        

    '''
    print("hi!")
    assert (test_type in v.TEST_TYPES)
    assert (data_type in v.DATA_TYPES)

    
    if data_type == "raw":
        dir = v.DIR_RAW
        data_key = 'Data'
    
        
    dataset = []
    counter = 0
    for f in os.listdir(dir):
        # go through for loop only if looking at backgroudn or task
        if test_type == "background" and not f.endswith("_1.mat"):
            continue
        if test_type == "task" and not f.endswith("_2.mat"):
            continue

        full_path = os.path.join(dir, f)
        
        print(f"Loading {f}...")
        mat = scipy.io.loadmat(full_path)
        key = [k for k in mat.keys() if not k.startswith("__")][0]
        data = mat[key]

        dataset.append(data)

        counter += 1

    dataset = np.array(dataset)
    print(f"Loaded {dataset.shape[0]} recordings from {dir} (shape per trial: {dataset.shape[1:]})")
    return dataset


def load_labels():
    '''
   TODO: Loads labels from the dataset and transforms the label values to binary values.

    Returns:
        ndarray: The labels.
    '''
    labels = pd.read_csv(v.LABELS_PATH)

    if 'Count quality' in labels.columns:
        label_array = labels['Count quality'].to_numpy().astype(int)

    print(f"Loaded {len(label_array)} labels (0=bad, 1=good)")
    return label_array


def format_labels(labels, test_type="task", epochs=1):
    '''
    Filter the labels and repeat for the specified amount of epochs.

    Args:
        labels (ndarray): The labels.
        test_type (string): The test_type to filter by. Defaults to "Arithmetic".
        epochs (int): The amount of epochs. Defaults to 1.

    Returns:
        ndarray: The formatted labels.

    '''
    assert test_type in v.TEST_TYPES

    return np.repeat(labels, epochs)


def split_data(data, sfreq):
    '''
    Splits EEG data into epochs with length 1 sec.

    Args:
        data (ndarray): EEG data.
        sfreq (int): The sampling frequency.
    
    Returns:
        ndarray: The epoched data.

    '''
    n_trials, n_channels, n_samples = data.shape
    n_epochs = n_samples // sfreq
    epoched = np.empty((n_trials, n_epochs, n_channels, sfreq))

    for i in range(n_trials):
        for j in range(n_epochs):
            epoched[i, j] = data[i, :, j*sfreq:(j+1)*sfreq]

    print(f"Split into {n_epochs} epochs per trial → shape: {epoched.shape}")
    return epoched


In [ ]:
# from datasetEEGMAT import load_dataset, load_labels, split_data, format_labels
# from featuresEEGMAT import time_series_features, fractal_features, entropy_features, hjorth_features, freq_band_features
import variablesEEGMAT as v
import datasetEEGMAT
import variablesEEGMAT
import featuresEEGMAT  

# reload after making edits to those files
importlib.reload(datasetEEGMAT)
importlib.reload(variablesEEGMAT)
importlib.reload(featuresEEGMAT)

<module 'featuresEEGMAT' from '/Users/prisharpatel/Desktop/ECE598/EECS598FinalProject/EEGMAT/featuresEEGMAT.py'>

# Variables

In [4]:
data_type = "raw"
test_type = "task"
data_dir = "data/converted"


# Load Dataset

In [ ]:
dataset_= load_dataset(data_type=data_type, test_type=test_type)
label_ = load_labels()
dataset = split_data(dataset_, v.SFREQ)
label = format_labels(label_, test_type=test_type, epochs=dataset.shape[1])

hi!
Loading Subject06_2.mat...
Loading Subject22_2.mat...
Loading Subject20_2.mat...
Loading Subject04_2.mat...
Loading Subject00_2.mat...
Loading Subject19_2.mat...
Loading Subject24_2.mat...
Loading Subject26_2.mat...
Loading Subject02_2.mat...
Loading Subject21_2.mat...
Loading Subject05_2.mat...
Loading Subject07_2.mat...
Loading Subject23_2.mat...
Loading Subject27_2.mat...
Loading Subject03_2.mat...
Loading Subject01_2.mat...
Loading Subject25_2.mat...
Loading Subject18_2.mat...
Loading Subject14_2.mat...
Loading Subject29_2.mat...
Loading Subject30_2.mat...
Loading Subject32_2.mat...
Loading Subject16_2.mat...
Loading Subject12_2.mat...
Loading Subject09_2.mat...
Loading Subject34_2.mat...
Loading Subject10_2.mat...
Loading Subject33_2.mat...
Loading Subject17_2.mat...
Loading Subject28_2.mat...
Loading Subject15_2.mat...
Loading Subject31_2.mat...
Loading Subject35_2.mat...
Loading Subject08_2.mat...
Loading Subject11_2.mat...
Loading Subject13_2.mat...
Loaded 36 recordings fro

# Compute Features

In [ ]:
# features = time_series_features(dataset)
# freq_bands = np.array([1, 4, 8, 12, 30, 50])
# features = freq_band_features(dataset, freq_bands)
# features = hjorth_features(dataset)
# features = entropy_features(dataset)
import numpy as np
import mne_features.univariate as mne_f
def fractal_features(data):
    """
    Computes the Higuchi and Katz fractal dimensions robustly.
    """

    n_trials, n_secs, n_channels, _ = data.shape
    features_per_channel = 2

    features = np.empty((n_trials, n_secs, n_channels * features_per_channel))

    # Normalize and sanitize
    data = np.nan_to_num(data, nan=0.0, posinf=0.0, neginf=0.0)
    data = data - np.mean(data, axis=-1, keepdims=True)
    data = data / (np.std(data, axis=-1, keepdims=True) + 1e-8)

    for i, trial in enumerate(data):
        for j, second in enumerate(trial):
            # Compute both features, catching numerical warnings
            try:
                higuchi = mne_f.compute_higuchi_fd(second)
            except Exception:
                higuchi = np.zeros(second.shape[0])

            try:
                katz = mne_f.compute_katz_fd(second)
                # Replace inf/nan with zeros or finite values
                katz = np.nan_to_num(katz, nan=0.0, posinf=0.0, neginf=0.0)
            except Exception:
                katz = np.zeros(second.shape[0])

            features[i, j] = np.concatenate([higuchi, katz])

    # Final cleanup
    features = np.nan_to_num(features, nan=0.0, posinf=0.0, neginf=0.0)
    print("NaNs after fractal features:", np.isnan(features).any())

    return features.reshape(n_trials * n_secs, n_channels * features_per_channel)

def time_series_features(data):
    '''
    Computes the features variance, RMS and peak-to-peak amplitude using the package mne_features.

    Args:
        data (ndarray): EEG data.

    Returns:
        ndarray: Computed features.

    '''

    n_trials, n_secs, n_channels, _ = data.shape
    features_per_channel = 3

    features = np.empty([n_trials, n_secs, n_channels * features_per_channel])
    for i, trial in enumerate(data):
        for j, second in enumerate(trial):
            variance = mne_f.compute_variance(second)
            rms = mne_f.compute_rms(second)
            ptp_amp = mne_f.compute_ptp_amp(second)
            features[i][j] = np.concatenate([variance, rms, ptp_amp])
    features = features.reshape(
        [n_trials*n_secs, n_channels*features_per_channel])
    return features

def freq_band_features(data, freq_bands):
    '''
    Computes the frequency bands delta, theta, alpha, beta and gamma using the package mne_features.

    Args:
        data (ndarray): EEG data.
        freq_bands (ndarray): The frequency bands to compute.

    Returns:
        ndarray: Computed features.
    '''
    n_trials, n_secs, n_channels, sfreq = data.shape
    features_per_channel = len(freq_bands)-1

    features = np.empty([n_trials, n_secs, n_channels * features_per_channel])
    for i, trial in enumerate(data):
        for j, second in enumerate(trial):
            psd = mne_f.compute_pow_freq_bands(
                sfreq, second, freq_bands=freq_bands)
            features[i][j] = psd
    features = features.reshape(
        [n_trials*n_secs, n_channels*features_per_channel])
    return features
def hjorth_features(data):
    '''
    Computes the features Hjorth mobility (spectral) and Hjorth complexity (spectral) using the package mne_features.

    Args:
        data (ndarray): EEG data.

    Returns:
        ndarray: Computed features.
    '''
    n_trials, n_secs, n_channels, sfreq = data.shape
    features_per_channel = 2

    features = np.empty([n_trials, n_secs, n_channels * features_per_channel])
    for i, trial in enumerate(data):
        for j, second in enumerate(trial):
            mobility_spect = mne_f.compute_hjorth_mobility_spect(sfreq, second)
            complexity_spect = mne_f.compute_hjorth_complexity_spect(
                sfreq, second)
            features[i][j] = np.concatenate([mobility_spect, complexity_spect])
    features = features.reshape(
        [n_trials*n_secs, n_channels*features_per_channel])
    return features

def entropy_features(data):
    '''
    Computes the features Approximate Entropy, Sample Entropy, Spectral Entropy and SVD entropy using the package mne_features.

    Args:
        data (ndarray): EEG data.

    Returns:
        ndarray: Computed features.

    '''
    n_trials, n_secs, n_channels, sfreq = data.shape
    features_per_channel = 4

    features = np.empty([n_trials, n_secs, n_channels * features_per_channel])
    for i, trial in enumerate(data):
        for j, second in enumerate(trial):
            app_entropy = mne_f.compute_app_entropy(second)
            samp_entropy = mne_f.compute_samp_entropy(second)
            spect_entropy = mne_f.compute_spect_entropy(sfreq, second)
            svd_entropy = mne_f.compute_svd_entropy(second)
            features[i][j] = np.concatenate(
                [app_entropy, samp_entropy, spect_entropy, svd_entropy])
    features = features.reshape(
        [n_trials*n_secs, n_channels*features_per_channel])
    return features

# features = fractal_features(dataset)
# features = time_series_features(dataset)
# freq_bands = np.array([1, 4, 8, 12, 30, 50])
# features = freq_band_features(dataset, freq_bands)
# features = hjorth_features(dataset)
features = entropy_features(dataset)
data = features


Contains inf: False
Contains nan: False


/Users/prisharpatel/Desktop/ECE598/EECS598FinalProject/.venv/lib/python3.13/site-packages/mne_features/univariate.py:1162: RuntimeWarning: invalid value encountered in divide
  psd_norm = np.divide(psd[:, 1:], m[:, None])
/Users/prisharpatel/Desktop/ECE598/EECS598FinalProject/.venv/lib/python3.13/site-packages/mne_features/univariate.py:1163: RuntimeWarning: divide by zero encountered in log2
  return -np.sum(np.multiply(psd_norm, np.log2(psd_norm)), axis=-1)
/Users/prisharpatel/Desktop/ECE598/EECS598FinalProject/.venv/lib/python3.13/site-packages/mne_features/univariate.py:1163: RuntimeWarning: invalid value encountered in multiply
  return -np.sum(np.multiply(psd_norm, np.log2(psd_norm)), axis=-1)
/Users/prisharpatel/Desktop/ECE598/EECS598FinalProject/.venv/lib/python3.13/site-packages/mne_features/univariate.py:1162: RuntimeWarning: invalid value encountered in divide
  psd_norm = np.divide(psd[:, 1:], m[:, None])
/Users/prisharpatel/Desktop/ECE598/EECS598FinalProject/.venv/lib/pyth

# k-NN Classifier

In [9]:
x, x_test, y, y_test = train_test_split(
    data, label, test_size=0.2, random_state=1)
x_train, x_val, y_train, y_val = train_test_split(
    x, y, test_size=0.25, random_state=1)
scaler = MinMaxScaler()
scaler.fit(x_train)
x = scaler.transform(x)
x_train = scaler.transform(x_train)
x_val = scaler.transform(x_val)
x_test = scaler.transform(x_test)

param_grid = {
    'leaf_size': range(50),
    'n_neighbors': range(1, 10),
    'p': [1, 2]
}
split_index = [-1 if x in range(len(x_train)) else 0 for x in range(len(x))]
ps = PredefinedSplit(test_fold=split_index)
knn_clf = GridSearchCV(KNeighborsClassifier(), param_grid, cv=ps, refit=True)
knn_clf.fit(x, y)

KeyboardInterrupt: 

In [136]:
y_pred = knn_clf.predict(x_test)
y_true = y_test


In [137]:
print(metrics.classification_report(y_true, y_pred))
print(metrics.confusion_matrix(y_true, y_pred))

              precision    recall  f1-score   support

           0       0.80      0.50      0.61       139
           1       0.81      0.94      0.87       308

    accuracy                           0.81       447
   macro avg       0.80      0.72      0.74       447
weighted avg       0.80      0.81      0.79       447

[[ 69  70]
 [ 17 291]]


# SVM Classifier

In [38]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV, PredefinedSplit

pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="mean")),
    ("svm", SVC())
])

x, x_test, y, y_test = train_test_split(
    data, label, test_size=0.2)
x_train, x_val, y_train, y_val = train_test_split(
    x, y, test_size=0.25)

param_grid = {
    'svm__C': [0.1, 1, 10, 100, 1000],
    'svm__kernel': ['rbf']
}
split_index = [-1 if x in range(len(x_train)) else 0 for x in range(len(x))]
ps = PredefinedSplit(test_fold=split_index)
svm_clf = GridSearchCV(pipeline, param_grid, cv=ps, refit=True)
svm_clf.fit(x, y)

,estimator,"Pipeline(step...svm', SVC())])"
,param_grid,"{'svm__C': [0.1, 1, ...], 'svm__kernel': ['rbf']}"
,scoring,None
,n_jobs,None
,refit,True
,cv,"PredefinedSpl...hape=(1785,)))"
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,missing_values,nan


In [39]:
y_pred = svm_clf.predict(x_test)
y_true = y_test

In [40]:
print(metrics.classification_report(y_true, y_pred))
print(metrics.confusion_matrix(y_true, y_pred))

              precision    recall  f1-score   support

           0       0.80      0.71      0.75       120
           1       0.90      0.94      0.92       327

    accuracy                           0.87       447
   macro avg       0.85      0.82      0.83       447
weighted avg       0.87      0.87      0.87       447

[[ 85  35]
 [ 21 306]]


# Multilayer Perceptron

In [141]:
K.clear_session()
y_v = label
y_v = to_categorical(y_v)
x_train, x_test, y_train, y_test = train_test_split(
    data, y_v, test_size=0.2, random_state=1)
x_train, x_val, y_train, y_val = train_test_split(
    x_train, y_train, test_size=0.25, random_state=1)

In [149]:
import keras

def model_builder(hp):
    model = models.Sequential()
    model.add(Input(shape=(x_train.shape[1],)))

    for i in range(hp.Int('layers', 2, 6)):
        model.add(Dense(units=hp.Int('units_' + str(i), 32, 1024, step=32),
                        activation=hp.Choice('act_' + str(i), ['relu', 'sigmoid'])))

    model.add(Dense(v.N_CLASSES, activation='softmax', name='out'))

    hp_learning_rate = hp.Choice('learning_rate', values=[1e-2, 1e-3, 1e-4])

    model.compile(optimizer=keras.optimizers.Adam(learning_rate=hp_learning_rate),
                  loss="binary_crossentropy",
                  metrics=['accuracy'])
    return model

In [150]:
tuner = RandomSearch(
    model_builder,
    objective='val_accuracy',
    max_trials=15,
    executions_per_trial=2,
    overwrite=True
)

In [151]:
tuner.search(x_train, y_train, epochs=50, validation_data=[x_val, y_val])

Trial 15 Complete [00h 00m 39s]
val_accuracy: 0.7337807416915894

Best val_accuracy So Far: 0.7393735945224762
Total elapsed time: 00h 08m 16s


In [152]:
model = tuner.get_best_models(num_models=1)[0]

/Users/prisharpatel/Desktop/ECE598/EECS598FinalProject/.venv/lib/python3.13/site-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 18 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [153]:
y_pred = model.predict(x_test)
y_true = y_test
y_pred = np.argmax(y_pred, axis=1)
y_true = np.argmax(y_true, axis=1)

14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step


In [154]:
print(metrics.classification_report(y_true, y_pred))
print(metrics.confusion_matrix(y_true, y_pred))

              precision    recall  f1-score   support

           0       0.68      0.24      0.36       139
           1       0.74      0.95      0.83       308

    accuracy                           0.73       447
   macro avg       0.71      0.60      0.59       447
weighted avg       0.72      0.73      0.68       447

[[ 34 105]
 [ 16 292]]
